# Getting started with ANTsTorch

Read the real `r16` image used in the [ANTsX tutorial](https://gist.github.com/ntustison/12a656a5fc2f6f9c4494c88dc09c5621), preserve its physical geometry
when converting to NumPy, and construct a PyTorch tensor. ANTsPy downloads the
sample on first use and reuses its cache. No simulated image is generated.

Install ANTsTorch and notebook tools from the repository root before opening this notebook:
```bash
python -m pip install -e '.[notebooks]'
python -m ipykernel install --user --name python3 --display-name "Python 3 (ANTsTorch)"
```
Select that kernel and run the cells in order. All examples default to CPU.

In [ ]:
import torch  # Import before ANTs to avoid OpenMP conflicts on macOS.
import numpy as np
import ants
import antstorch

# Display figures inside Jupyter/VS Code even if MPLBACKEND was set to Agg.
from IPython import get_ipython
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

device = "cpu"
antstorch.set_default_device(device)

## An image is more than an array
ANTs images carry spacing, origin and direction. NumPy arrays do not retain these
automatically. Here the first array axis is the physical x axis.

In [ ]:
image = ants.image_read(ants.get_ants_data("r16")).clone("float")
array = image.numpy()
reconstructed = ants.from_numpy(array, spacing=image.spacing,
                                origin=image.origin, direction=image.direction)
assert reconstructed.shape == image.shape
np.testing.assert_allclose(reconstructed.numpy(), array)
np.testing.assert_allclose(reconstructed.direction, image.direction)
assert reconstructed.spacing == image.spacing
assert reconstructed.origin == image.origin
print(image)

## Convert spatial axes for tensor operations
A scalar 2-D tensor has layout `(batch, channels, height, width)`.
ANTs/ITK uses `(x, y)`; transpose to `(y, x)` and retain the geometry separately.

In [ ]:
tensor = torch.from_numpy(image.numpy().T.copy())[None, None].to(device)
assert tuple(tensor.shape) == (1, 1, *reversed(image.shape))
assert torch.isfinite(tensor).all()
plt.imshow(tensor[0, 0].cpu(), origin="lower", cmap="gray")
plt.title("Tutorial r16 (voxel coordinates)")
plt.xlabel("x voxel"); plt.ylabel("y voxel")
plt.show()

Continue with `unet_2d.ipynb`, `n4_bias_correction.ipynb`, or
`syn_registration.ipynb`. The pretrained application notebooks download data,
templates and weights on first use; inspect their resource notes first.